# VibroPredict: Physics-Aware Enzyme Kinetics Prediction

This notebook provides an overview of the VibroPredict system, which predicts enzyme catalytic turnover (k_cat) using three modalities:
1. **Sequence**: ProtT5 protein language model embeddings
2. **Spectral**: Vibrational Density of States (VDOS) from Normal Mode Analysis
3. **Chemical**: Substrate SMILES + Differential Reaction Fingerprints (DRFP)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import torch

# VibroPredict imports
from vibropredict.models.vibropredict_hybrid import VibroPredictHybrid
from vibropredict.training.metrics import compute_all_metrics

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

## Architecture Overview

```
Sequence ──► ProtT5 Encoder ──► h_seq (1024-dim)
                                      │
VDOS ──────► SpectralCNN ─────► h_spec (128-dim)  ──► TriModalFusion ──► MLP ──► log₁₀(k_cat)
                                      │
SMILES ────► ChemBERTa+DRFP ──► h_chem (1024-dim)
```

In [ ]:
# Initialize model (downloads ProtT5 and ChemBERTa on first run)
model = VibroPredictHybrid(
    spec_dim=128,
    seq_dim=1024,
    chem_dim=1024,
    fusion_dim=512,
    dropout=0.2,
)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

In [ ]:
# Example inference with dummy data
sequences = ['MKTIIALSYIF', 'ACDEFGHIKLMNPQRSTVWY']
vdos = torch.randn(2, 1, 1000)  # 2 proteins, 1000-bin VDOS
substrate_smiles = ['CC(=O)O', 'c1ccccc1']

with torch.no_grad():
    logkcat, gates = model(sequences, vdos, substrate_smiles)

print(f'Predicted log10(k_cat): {logkcat.numpy()}')
print(f'Attention gates (seq, spec, chem): {gates.numpy()}')

## Next Steps

1. **Database Construction**: See `06_database_construction.ipynb`
2. **Model Training**: See `07_model_training.ipynb`
3. **Ablation Study**: See `08_ablation_study.ipynb`